### READ DATA

In [0]:
file_location = "/Volumes/workspace/default/sample_data/big_mar_seles.csv"
file_type = "csv"
df = spark.read.format(file_type) \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .load(file_location) 

In [0]:
df.display()

In [0]:
df.columns

### Schema difination

In [0]:
df.printSchema()

### change type of futer

- ###### we change  Item_Weight from double to string

In [0]:
my_ddl_schema = '''  
                    tem_Identifier string,
                    Item_Weight string,
                    Item_Fat_Content string,
                    Item_Visibility double,
                    Item_Type string,
                    Item_MRP double,
                    Outlet_Identifier string,
                    Outlet_Establishment_Year integer,
                    Outlet_Size string,
                    Outlet_Location_Type string,
                    Outlet_Type string,
                    Item_Outlet_Sales double'''

In [0]:
df = spark.read.format(file_type).schema(my_ddl_schema).option("header", "true") \
  .option("inferSchema", "true") \
  .load(file_location) 

In [0]:
df.printSchema()    

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
my_strct_shema = StructType([
  StructField("tem_Identifier", StringType(), True),
  StructField("Item_Weight", StringType(), True),
  StructField("Item_Fat_Content", StringType(), True),
  StructField("Item_Visibility", StringType(), True),
  StructField("Item_Type", StringType(), True),
  StructField("Item_MRP", StringType(), True),
  StructField("Outlet_Identifier", StringType(), True),
  StructField("Outlet_Establishment_Year", StringType(), True),
  StructField("Outlet_Size", StringType(), True),
  StructField("Outlet_Location_Type", StringType(), True),
  StructField("Outlet_Type", StringType(), True),
  StructField("Item_Outlet_Sales", StringType(), True)
])

df = spark.read.format(file_type).schema(my_strct_shema).option("header", "true") \
  .option("inferSchema", "true") \
  .load(file_location) 
df.printSchema()

## select function

In [0]:
df_select = df.select("Item_Identifier", "Item_Weight", "Item_Fat_Content", "Item_Visibility")


df_select.display()

## Alias  is like (as) in sql

In [0]:
df.select(col('Item_Identifier').alias('Item_ID')).display()



## FILTER it is (WHERE) in sql

##### SCENARIO 1

In [0]:
df.display()

In [0]:
df.filter(col('Item_Weight') >21)

###### SCENARIO 2

In [0]:
df.filter((col('Item_Weight') >21) & (col('Item_Fat_Content') == 'LF')).display()

##### SCENARIO 3

In [0]:
df2 = df.filter((col('Outlet_Size').isNull()) & (col('outlet_location_type').isin('Tier 1', 'Tier 2')))

df2.select('Outlet_Size','Outlet_Location_Type').display()


### withcolummnRenamed 

##### (chane name of the column)

In [0]:
df.withColumnRenamed('Item_Identifier','Item_ID').display()

### wothColumn

##### add a new column

In [0]:
df = df.withColumn('flag',lit('new'))
df.display()

In [0]:
df = df.withColumn('multiply',col('Item_Weight') * col('Item_MRP'))

df.display()

##### replace null value with Medium

In [0]:
df = df.withColumn('Outlet_Size',when(col('Outlet_Size').isNull(),'Medium'))

df.display()

##### replace (Low Fat) with (LF) and Regular wit Rg

In [0]:
df = df. withColumn('Item_Fat_Content',regexp_replace('Item_Fat_Content','Low Fat','LF')) \
        .withColumn('Item_Fat_Content',regexp_replace('Item_Fat_Content', 'Regular', 'Rg'))

df.display()

Type Costing  cost() chang etype of column

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

df.printSchema()

### sort() it si orederby 

In [0]:
df.sort(col('Item_weight').asc()).display()

In [0]:
df.sort(col('Item_weight').desc()).display()

2 columns desc

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[False,False]).display()

In [0]:
df.sort(['Item_Weight','Item_Visibility'], ascending=[True,False]).display()

### LIMIT function

In [0]:
df.limit(10).display()

### Drop function

In [0]:
df.drop('Item_Weight').display()

In [0]:
df.drop('Item_Fat_Content','Item_Visibility','Item_Type').display()


### dropDuplicates


In [0]:
df.dropDuplicates().display()

##### drop duplicate in one or more 

In [0]:
df.drop_duplicates(subset=['Item_Weight']).display()

##### remove duplicate like function dropDuplicate()

In [0]:
df.select("Item_Weight").distinct().display()

### UNION and UNION BY NAME

##### preparin the dataframe

In [0]:

data1 = [('1','kad'),
        ('2','sid')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','rahul'),
        ('4','jas')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
df1.union(df2).display()

In [0]:

data2 = [('rahul','3'),
        ('jas','4')]
schema2 = 'name STRING, id STRING' 

df2 = spark.createDataFrame(data2,schema2)

##### runing by column Name or futer 

In [0]:
df1.unionByName(df2).display()

### String Functions

####### at beginning of each word the first latter becomes capitalized


In [0]:
df.select(initcap('Item_Type')).display()

In [0]:
df.select(lower('Item_Type')).display()

In [0]:
df.select(upper('Item_Type')).display()

In [0]:
df.select(lower('Item_Type').alias('upper_item_type')).display()

### DATE FUNCTIONS

#### current_date()

In [0]:
df=df.withColumn('Curr_Date',current_date())
df.display()

#### date_add()

######## we days on the time we have before on this example we add 6 days

In [0]:
df = df.withColumn('week_after',date_add('Curr_date',7))
df.display()


#### date_sub()

### we substract from (curr_date)  7 days 

In [0]:
df = df.withColumn('week_before',date_sub('Curr_date',7))
df.display()

#### datediff(after,before) becuse the calculate is before-after

In [0]:
df = df.withColumn('datediff',datediff('week_after','Curr_Date'))
df.display()

#### date_format() cahnge formate of date

In [0]:
df = df. withColumn('date_format',date_format('week_after','dd-MM-yyyy'))
df.display()

### HUNDLING NULLS

#### drona()

(all) remove only te record oe row his have all the columns null

In [0]:
df.dropna('all').display()

##### (any) it's removing any recorde have null on his column

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset='Item_Weight').display()

### fillna()

In [0]:
df.fillna('notavaliable').display()

In [0]:
df.fillna(0).display()

In [0]:
df.fillna(0,subset='Item_Weight').display()

### split and indexing

In [0]:
df2 = df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df2.display()

In [0]:
df.select(split('Item_Type',' ')).display()

#### indexing

In [0]:
df.select(split('Item_Type',' ')[0]).display()

### EXPLODE

In [0]:
df2.withColumn('Outlet_Type',explode('Outlet_Type')).display()

### array_contains

In [0]:
df2.withColumn('Type_Flag',array_contains('Outlet_Type','Type1')).display()

### groupBy

In [0]:
df.groupBy("Item_Type").agg(sum('Item_Outlet_Sales')).display()

In [0]:
df.groupBy('Item_Type').count().display()

In [0]:
df.groupBy('Item_Type').agg(count('*').alias('count_of_types')).display()

In [0]:
df.groupBy('Item_Type').agg(avg('Item_Outlet_Sales').alias('avg_sales')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum("Item_MRP").alias('total_MRP')).display()

In [0]:
df.groupBy('Item_Type','Outlet_Size').agg(sum("Item_MRP").alias('Totale_MRP'),avg('Item_MRP').alias('avg_MRP')).display()

### collect_list

In [0]:

data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
df_book.groupBy('user').agg(collect_list('book').alias('books')).display()

### pivot

In [0]:
df3 = df.select('Item_Type','Outlet_Size','Item_MRP')
df3.display()

In [0]:
df3.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

### when().otherwise()

In [0]:
df.display()

In [0]:
df_f = df.withColumn('veg_flag',when(col('Item_Type')=='Meat','non-veg').otherwise('veg'))
df_f.display()

In [0]:
df_v = df_f.withColumn('veg_exp_flag', when((col('veg_flag') =='veg') & (col('Item_MRP') > 100),'veg-expensive')\
                              .when((col('veg_flag') =='veg') & (col('Item_MRP') <= 100),'veg-cheap')\
                              .when((col('veg_flag') =='non-veg') & (col('Item_MRP') > 100),'non-veg-expensive')\
                              .when((col('veg_flag') =='non-veg') & (col('Item_MRP') <= 100),'non-veg-cheap'))
df_v.display()

### join

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()
df2.display()

In [0]:
df1.join(df2,df1['dept_id'] == df2['dept_id'],'inner').display()

In [0]:
df1.join(df2,df1['dept_id'] == df2['dept_id'],'left').display()

In [0]:
df1.join(df2,df1['dept_id'] == df2['dept_id'],'right').display()

In [0]:
df1.join(df2,df1['dept_id'] == df2['dept_id'],'anti').display()

### WINDOW FUNCTIONS

##### ROW_NUMBER()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

##### RANK VS DENSE RANK

In [0]:
df1 = df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
        .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc())))
df1.display() 

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

##### Cumulative Sum

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type'))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('totalsum',sum('Item_MRP').over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

##### USER DEFINED FUNCTIONS (UDF)

In [0]:
def my_func(x):
    return x*x 

In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('mynewcol',my_udf('Item_MRP')).display()

### DATA WRITING

##### csv

In [0]:
df.write.format('csv')\
        .save('/FileStore/tables/CSV/data.csv')

##### append

In [0]:
df.write.format('csv')\
        .mode('append')\
        .save('/FileStore/tables/CSV/data.csv')

In [0]:
df.write.format('csv')\
        .mode('append')\
        .option('path','/FileStore/tables/CSV/data.csv')\
        .save()

##### overwrite

In [0]:
df.write.format('csv')\
.mode('overwrite')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()

##### error

In [0]:
df.write.format('csv')\
.mode('error')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()

##### ignore

In [0]:
df.write.format('csv')\
.mode('ignore')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()

##### parquet

In [0]:

df.write.format('parquet')\
.mode('overwrite')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()
     

##### table

In [0]:
df.write.format('parquet')\
.mode('overwrite')\
.saveAsTable('my_table')
df.display()

### SPARK SQL

##### createTempView

In [0]:
df.createTempView('my_view')

In [0]:
%sql
select * from my_view where Item_Fat_Content = 'Lf'

In [0]:
df_sql = spark.sql("select * from my_view where Item_Fat_Content = 'Lf'")

In [0]:
df_sql.display()